# zynarai v1 — ACE-Step 1.5 LoKr Training (Colab Pro)

**Before running:** Runtime → Change runtime type → **L4 GPU** (24 GB, Ada — don't accept a T4,
it's below the Flash Attention floor). Upload your local `dataset/` folder (audio/ + dataset.json,
~2.5 GB) to Drive at `MyDrive/zynarai-lora/dataset/` before starting.

**Colab discipline baked into this notebook:** preprocessed tensors and training checkpoints
are written to Drive, so a disconnect never costs you more than the current epoch. Re-running
the notebook top-to-bottom skips completed stages.

Pipeline: GPU check → mount Drive → install ACE-Step-1.5 → download base model →
preprocess (cached) → optional estimate → **LoKr training** → Gradio UI to hear results.

In [ ]:
# 1 — GPU check: want L4 (24GB) or A100. T4 = reconnect and try again.
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
cc = torch.cuda.get_device_capability()
assert cc >= (8, 0), f"compute capability {cc} < 8.0 — no Flash Attention. Get an L4/A100 runtime."
print("GPU OK")

In [ ]:
# 2 — Mount Drive + define layout
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE   = Path('/content/drive/MyDrive/zynarai-lora')
DATASET = DRIVE / 'dataset'          # you uploaded this (audio/ + dataset.json)
TENSORS = DRIVE / 'tensors'          # preprocessing cache (written once)
OUTPUT  = DRIVE / 'output/lokr_v1'   # training checkpoints land here (disconnect-safe)
for p in (TENSORS, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)
assert (DATASET / 'dataset.json').exists(), 'upload dataset/ to MyDrive/zynarai-lora/ first'
n_audio = len(list((DATASET / 'audio').glob('*')))
print(f'dataset found: {n_audio} audio files')

In [ ]:
# 3 — Install ACE-Step 1.5 + Side-Step deps (~5 min first run)
%cd /content
![ -d ACE-Step-1.5 ] || git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!pip install -q -r requirements.txt
!pip install -q -r requirements-sidestep.txt
# flash-attn is optional — trainer falls back to SDPA if this fails; don't fight it
!pip install -q flash-attn --no-build-isolation || echo 'flash-attn skipped (SDPA fallback)'

In [ ]:
# 4 — Download base model checkpoints (~20GB to fast local disk; re-downloads per session)
CKPT = '/content/checkpoints'
from huggingface_hub import snapshot_download
snapshot_download('ACE-Step/Ace-Step1.5', local_dir=CKPT)
!ls {CKPT}

In [ ]:
# 5 — Copy dataset from Drive to local disk (Drive I/O is slow for audio decode)
!mkdir -p /content/data && cp -rn {DATASET}/. /content/data/
!ls /content/data/audio | head -3 && ls /content/data/audio | wc -l

In [ ]:
# 6 — Preprocess audio -> training tensors (skipped if the Drive cache already has them)
import os
if any(Path(str(TENSORS)).iterdir()):
    print('tensor cache found on Drive — skipping preprocessing')
else:
    !python train.py fixed \
        --checkpoint-dir {CKPT} \
        --model-variant base \
        --preprocess \
        --audio-dir /content/data/audio \
        --dataset-json /content/data/dataset.json \
        --tensor-output {TENSORS}
!ls {TENSORS} | wc -l

In [ ]:
# 7 (optional) — Gradient sensitivity estimate (~2 min): which modules matter most
!python train.py estimate \
    --checkpoint-dir {CKPT} \
    --model-variant base \
    --dataset-dir {TENSORS} \
    --estimate-batches 5 --top-k 16

In [ ]:
# 8 — LoKr training. 57 tracks -> Side-Step guidance is 50-100 epochs for 50+ songs.
#     Checkpoints save to Drive every 10 epochs; resume with --resume-from if disconnected.
!python train.py fixed \
    --checkpoint-dir {CKPT} \
    --model-variant base \
    --dataset-dir {TENSORS} \
    --output-dir {OUTPUT} \
    --adapter-type lokr \
    --lokr-linear-dim 64 \
    --lokr-linear-alpha 128 \
    --lokr-factor -1 \
    --lokr-weight-decompose \
    --learning-rate 0.03 \
    --epochs 100 \
    --batch-size 2 \
    --gradient-accumulation 2 \
    --save-every 10 \
    --seed 42

**If the session disconnects mid-run:** re-run cells 1–6 (preprocessing will skip via the
Drive cache), then re-run cell 8 with `--resume-from` pointing at the latest checkpoint in
`MyDrive/zynarai-lora/output/lokr_v1/`.

**Smoke test first!** For the very first run, change `--epochs 100` to `--epochs 5` and
confirm the whole pipeline completes and loss decreases before committing to the full run.

In [ ]:
# 9 — Hear it: launch the ACE-Step Gradio UI with a public link.
#     In the UI: Service Configuration -> load model (base) -> load your LoKr adapter from
#     the output dir -> prompt with the trigger word, A/B with and without it:
#       "zynarai, driving tech house, gritty bassline pressure, dark moody tone"
#       "driving tech house, gritty bassline pressure, dark moody tone"   (control)
import os
os.environ['ACESTEP_CHECKPOINT_DIR'] = CKPT
!acestep --share || python -m acestep.api_server --share

## Evaluation checklist (from PLAN.md phase 4)

1. Fixed 10-prompt A/B: trigger word vs no trigger word, same seeds.
2. Overfit check: does it regurgitate recognizable melodies from the 57 (bad), or capture
   texture/groove/tonality (good)? Compare epoch-50 vs epoch-100 checkpoints — earlier
   often generalizes better.
3. Dilution reality-check: given the v1 captions, expect `underground warehouse energy` and
   `clean modern club mix` to do little (they're near-constants); the axes that stayed
   contrastive (`weighty kick-driven low end`, `busy layered percussion`, R&B hooks) are the
   dials most worth testing.
4. Download the final adapter from Drive; it also runs locally in the macOS Gradio UI.
5. Write up labeling-effort + quality notes -> feeds the v2 (Zynar) plan and the startup thesis.